# 12.4 Other output commands: print and printf

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Two additional AMPL commands have much the same syntax as `display`, but do
not automatically format their output. The print command does no formatting at all,
while the printf command requires an explicit description of the desired formatting.

**The print command**

A print command produces a single line of output:

```ampl
ampl: print sum {p in PROD, t in 1..T} revenue[p,t]*Sell[p,t],
ampl?       sum {p in PROD, t in 1..T} prodcost[p]*Make[p,t],
ampl?       sum {p in PROD, t in 1..T} invcost[p]*Inv[p,t];
787810 269477 3300
ampl: print {t in 1..T, p in PROD} Make[p,t];
5990 1407 6000 1400 1400 3500 2000 4200
```

or, if followed by an indexing expression and a colon, a line of output for each member of
the index set:

```ampl
ampl: print {t in 1..T}: {p in PROD} Make[p,t];
5990 1407
6000 1400
1400 3500
2000 4200
```

Printed entries are normally separated by a space, but option print_separator can
be used to change this. For instance, you might set print_separator to a tab for
`data` to be imported by a spreadsheet; to do this, type option print_separator
"→", where → stands for the result of pressing the tab key.

The keyword print (with optional indexing expression and colon) is followed by a
print item or comma-separated list of print items. A print item can be a value, or an
indexing expression followed by a value or parenthesized list of values. Thus a print item
is much like a `display` item, except that only individual values may appear; although you
can say `display` rate, you must explicitly specify print {p in PROD} rate[p].
Also a set may not be an argument to print, although its members may be:

```ampl
ampl: print PROD;
syntax error
context: print >>> PROD; <<<

ampl: print {p in PROD} (p, rate[p]);
bands 200 coils 140
```

Unlike `display`, however, print allows indexing to be nested within an indexed item:

```ampl
ampl: print {p in PROD} (p, rate[p], {t in 1..T} Make[p,t]);
bands 200 5990 6000 1400 2000 coils 140 1407 1400 3500 4200
```

The representation of numbers in the output of print is governed by the
print_precision and print_round options, which work exactly like the
display_precision and display_round options for the `display` command.
Initially print_precision is 0 and print_round is an empty string, so that by
default print uses as many digits as necessary to represent each value as precisely as
possible. For the above examples, print_round has been set to 0, so that the numbers
are rounded to integers.

Working interactively, you may find print useful for viewing a few values on your
screen in a more compact format than `display` produces. With output redirected to a
file, print is useful for writing unformatted results in a form convenient for spreadsheets
and other `data` analysis tools. As with `display`, just add >filename to the end of
the print command.

**The printf command**

The syntax of printf is exactly the same as that of print, except that the first
print item is a character string that provides formatting instructions for the remaining
items:

```ampl
ampl: printf "Total revenue is $%6.2f.\n",
ampl?    sum {p in PROD, t in 1..T} revenue[p,t]*Sell[p,t];
Total revenue is $787810.00.
```

The format string contains two types of objects: ordinary characters, which are copied to
the output, and conversion specifications, which govern the appearance of successive
remaining print items. Each conversion specification begins with the character % and
ends with a conversion character. For example, %6.2f specifies conversion to a decimal
representation at least six characters wide with two digits after the decimal point. The
complete rules are much the same as for the printf function in the C programming
language; a summary appears in Section A.16 of the Appendix.

The output from printf is not automatically broken into lines. A line break must
be indicated explicitly by the combination \n, representing a "newline" character, in the
format string. To produce a series of lines, use the indexed version of printf:

```ampl
ampl:    printf {t in 1..T}: "%3i%12.2f%12.2f\n", t,
ampl?       sum {p in PROD} revenue[p,t]*Sell[p,t],
ampl?       sum {p in PROD} prodcost[p]*Make[p,t];
 1      159210.00    75377.00
 2      243500.00    75400.00
 3      167300.00    52500.00
 4      217800.00    66200.00
```

This printf is executed once for each member of the indexing set preceding the colon;
for each `t` in 1..T the format is applied again, and the \n character generates another
line break.

The printf command is mainly useful, in conjunction with redirection of output to
a file, for printing short summary reports in a readable format. Because the number of
conversion specifications in the format string must match the number of values being
printed, printf cannot conveniently produce tables in which the number of items on a
line may vary from run to run, such as a `table` of all Make[p,t] values.